# Step 4 (Baseline EfficientAD-S B03) — Kaggle GPU Training (30h Free / Chạy ngầm Background)

Notebook này được thiết kế tối ưu riêng cho **Kaggle Notebooks**, tận dụng GPU T4 (30 giờ/tuần) và tính năng chạy ngầm (**Save Version ▸ Save & Run All**) để huấn luyện trọn vẹn 70 epoch của EfficientAD-S mà bạn không cần phải mở máy tính hay canh chừng.

---

### Hướng dẫn chuẩn bị trên Kaggle (3 bước đơn giản):
1. **Tạo Kaggle Dataset:**
   - Nén thư mục repo `PCB_Lab_impl4` thành `PCB_Lab_impl4.zip`.
   - Đảm bảo bạn có file `DatasetVer4_Public.zip`.
   - Vào **Kaggle ▸ Datasets ▸ New Dataset**, tải cả 2 file ZIP lên và đặt tên dataset (ví dụ: `pcb-efficientad-lab`).
2. **Tạo Notebook trên Kaggle:**
   - Vào **Kaggle ▸ Code ▸ New Notebook**.
   - Chọn menu **File ▸ Upload Notebook** và tải file `train_efficientad_kaggle.ipynb` này lên.
   - Ở cột bên phải (Notebook Settings):
     - **Accelerator:** Chọn **GPU T4 x 1** (hoặc P100).
     - **Internet:** Bật **Internet ON** (Bắt buộc để tải teacher pretrained weights).
     - **Input (Add Data):** Chọn dataset bạn vừa tạo ở bước 1.
3. **Chạy ngầm qua đêm:**
   - Bấm nút **Save Version** (góc trên bên phải) ▸ Chọn **Save & Run All (Commit)** ▸ Bấm **Save**.
   - Bạn có thể **tắt trình duyệt, tắt máy tính**. Kaggle sẽ tự động chạy ngầm toàn bộ 70 epoch trong khoảng 1.5 tiếng.
   - Khi chạy xong, vào tab **Output** của notebook trên Kaggle để tải 2 file `efficientad_step4_artifacts.zip` và `efficientad_step4_runs.zip` về máy.

In [ ]:
# ===== Cell 1: Hardware & Environment Verification =====
import os, sys, subprocess

def banner(t):
    print('=' * 60)
    print(t)
    print('=' * 60)

banner('Kiểm tra GPU và cấu hình phần cứng Kaggle')
!nvidia-smi || echo 'Không tìm thấy lệnh nvidia-smi'

import torch
cuda_ok = torch.cuda.is_available()
print('PyTorch version:', torch.__version__)
print('CUDA available:', cuda_ok)
if cuda_ok:
    name = torch.cuda.get_device_name(0)
    cap = torch.cuda.get_device_capability(0)
    print('GPU Device:', name, f'(Compute Capability: {cap})')
    assert cap[0] >= 7, f'Compute capability {cap} quá thấp để chạy mô hình.'
    print('--> Phần cứng GPU hợp lệ, sẵn sàng huấn luyện.')
else:
    raise SystemExit('LỖI: Chưa bật GPU! Vui lòng vào Settings bên phải -> Accelerator -> Chọn GPU T4 x 1.')

In [ ]:
# ===== Cell 2: Tự động nhận diện Dataset (Read-Only từ /kaggle/input) =====
import os, shutil, zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
print('Tìm kiếm dataset trong /kaggle/input...')

# 1. Tìm DATASET_CARD.md (dấu hiệu nhận diện duy nhất của DatasetVer4_Public)
card_files = [f for f in input_root.rglob('DATASET_CARD.md') if 'impl4' not in str(f).lower()]

if card_files:
    dataset_root = card_files[0].parent
    print(f'--> [TÌM THẤY] Dataset DeepPCB tại: {dataset_root}')
else:
    # Fallback: tìm samples.jsonl (loại trừ các thư mục cache/runs/test/impl4)
    manifest_files = [
        f for f in input_root.rglob('samples.jsonl')
        if not any(k in str(f).lower() for k in ['runs', 'smoke', 'test', 'impl4'])
    ]
    if manifest_files:
        dataset_root = manifest_files[0].parents[3]
        print(f'--> [TÌM THẤY] Dataset DeepPCB qua samples.jsonl tại: {dataset_root}')
    else:
        # Nếu người dùng để nguyên tệp ZIP trong /kaggle/input
        zip_files = list(input_root.rglob('*[Dd]ataset*.zip')) + list(input_root.rglob('*[Dd]eep[Pp][Cc][Bb]*.zip'))
        if zip_files:
            target_zip = zip_files[0]
            print(f'--> Giải nén tệp dataset zip: {target_zip.name} vào /kaggle/working/...')
            dataset_root = Path('/kaggle/working/DatasetVer4_Public')
            dataset_root.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(target_zip, 'r') as zf:
                zf.extractall('/kaggle/working/ds_temp')
            sub_card = list(Path('/kaggle/working/ds_temp').rglob('DATASET_CARD.md'))
            if sub_card:
                sub_root = sub_card[0].parent
            else:
                sub_manifest = list(Path('/kaggle/working/ds_temp').rglob('samples.jsonl'))
                if sub_manifest:
                    sub_root = sub_manifest[0].parents[3]
                else:
                    raise FileNotFoundError('Không tìm thấy samples.jsonl trong tệp zip dataset!')
            if dataset_root.exists():
                shutil.rmtree(dataset_root)
            shutil.move(str(sub_root), str(dataset_root))
            shutil.rmtree('/kaggle/working/ds_temp', ignore_errors=True)
        else:
            raise FileNotFoundError('Không tìm thấy Dataset trong /kaggle/input. Vui lòng bấm Add Data ở góc phải!')

# Kiểm tra tính hợp lệ của dataset_root
samples_path = dataset_root / 'benchmarks' / 'deeppcb' / 'manifests' / 'samples.jsonl'
assert samples_path.exists(), f'LỖI: Không tìm thấy samples.jsonl tại {samples_path}!'

os.environ['DATASET_ROOT'] = str(dataset_root)
print('--> [HỢP LỆ] DATASET_ROOT đã đặt thành:', os.environ['DATASET_ROOT'])


In [ ]:
# ===== Cell 3: Thiết lập Codebase tại /kaggle/working =====
import os, sys, shutil, zipfile
from pathlib import Path

input_root = Path('/kaggle/input')
working_dir = Path('/kaggle/working')
local_repo = working_dir / 'PCB_Lab_impl4'

print('Tìm kiếm codebase PCB_Lab_impl4 trong /kaggle/input...')

# 1. Tìm file train_efficientad.py để xác định chính xác thư mục repo (tránh nhầm với DatasetVer4) --max-steps 70000
script_files = list(input_root.rglob('train_efficientad.py')) --max-steps 70000

if script_files:
    # script_files[0] là .../scripts/train_efficientad.py -> repo root là parent thứ 2 --max-steps 70000
    source_repo = script_files[0].parent.parent
    print(f'--> [TÌM THẤY] Source code tại: {source_repo}')
    if local_repo.exists():
        shutil.rmtree(local_repo)
    print(f'--> Đang sao chép mã nguồn sang {local_repo} để có quyền ghi...')
    shutil.copytree(
        source_repo,
        local_repo,
        ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', '.venv', '*.pt')
    )
else:
    # Nếu người dùng để nguyên tệp ZIP repo mà chưa được Kaggle giải nén
    repo_zips = [z for z in input_root.rglob('*.zip') if 'impl4' in z.name.lower()]
    if not repo_zips:
        repo_zips = [z for z in input_root.rglob('*.zip') if 'pcb' in z.name.lower() and 'dataset' not in z.name.lower()]
    if repo_zips:
        target_zip = repo_zips[0]
        print(f'--> Giải nén tệp repo zip: {target_zip.name}...')
        temp_repo = working_dir / 'temp_repo'
        if temp_repo.exists():
            shutil.rmtree(temp_repo)
        with zipfile.ZipFile(target_zip, 'r') as zf:
            zf.extractall(temp_repo)
        sub_script = list(temp_repo.rglob('train_efficientad.py')) --max-steps 70000
        if sub_script:
            source_repo = sub_script[0].parent.parent
            if local_repo.exists():
                shutil.rmtree(local_repo)
            shutil.copytree(
                source_repo,
                local_repo,
                ignore=shutil.ignore_patterns('.git', '__pycache__', '.pytest_cache', '.venv', '*.pt')
            )
            shutil.rmtree(temp_repo, ignore_errors=True)
        else:
            raise FileNotFoundError('Không tìm thấy train_efficientad.py trong file zip repo!') --max-steps 70000
    else:
        raise FileNotFoundError('Không tìm thấy mã nguồn PCB_Lab_impl4 trong /kaggle/input!')

# 2. Xác minh tính toàn vẹn của repo vừa copy
train_script = local_repo / 'scripts' / 'train_efficientad.py' --max-steps 70000
extract_script = local_repo / 'scripts' / 'extract_efficientad_predictions.py'
assert train_script.exists(), f'LỖI: Không tìm thấy {train_script}!'
assert extract_script.exists(), f'LỖI: Không tìm thấy {extract_script}!'
print(f'--> [XÁC MINH HỢP LỆ] Đã kiểm tra đầy đủ scripts tại: {local_repo / "scripts"}')

# 3. Chuyển thư mục làm việc vào local_repo
os.chdir(str(local_repo))
print('--> Thư mục làm việc hiện tại:', os.getcwd())

# 4. Đặt PYTHONPATH
src_path = str(local_repo / 'src')
sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = f"{src_path}:{str(local_repo)}:{os.environ.get('PYTHONPATH', '')}"
print('--> PYTHONPATH đã cấu hình xong.')


In [ ]:
# ===== Cell 4: Cài đặt thư viện bổ sung (Giữ nguyên PyTorch CUDA của Kaggle) =====
import subprocess, sys

# Kaggle đã cài sẵn PyTorch và CUDA 12.x rất chuẩn. Ta chỉ cài các thư viện phụ trợ:
!pip install -q opencv-python-headless==4.11.0.86 pyyaml==6.0.3 psutil==7.0.0

import torch
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('Device Name:', torch.cuda.get_device_name(0))

In [ ]:
# ===== Cell 5: Huấn luyện EfficientAD-S (B03) — max_steps=70,000 (~8.7 Epochs) =====
!python scripts/train_efficientad.py --config configs/models/efficientad_s.yaml --seed 42 --device 0 --allow-download --max-steps 70000

In [ ]:
# ===== Cell 6: Trích xuất dự đoán (Calibration & Fusion) =====
!python scripts/extract_efficientad_predictions.py --artifact artifacts/efficientad/B03_efficientad_s/seed42 --device 0

In [ ]:
# ===== Cell 7: Đối soát số lượng dòng dự đoán và Artifacts =====
import json
from pathlib import Path

artifact = Path('artifacts/efficientad/B03_efficientad_s/seed42')
assert artifact.exists(), 'Thư mục artifact không tồn tại!'

meta = json.loads((artifact / 'artifact.json').read_text(encoding='utf-8'))
print('Model ID:', meta['model_id'])
print('Checkpoint SHA256:', meta['checkpoint_sha256'])
print('Smoke flag:', meta['smoke'])
assert meta['smoke'] is False, 'Lỗi: Artifact bị gắn cờ smoke!'

for split, expected in [('calibration', 460), ('fusion', 306)]:
    pred_file = artifact / f'preds_{split}.jsonl'
    assert pred_file.exists(), f'Thiếu file {pred_file}!'
    lines = pred_file.read_text(encoding='utf-8').splitlines()
    rows = [json.loads(line) for line in lines]
    assert len(rows) == expected, f'Số dòng {split} ({len(rows)}) không khớp {expected}!'
    assert len({r['sample_id'] for r in rows}) == expected, f'Trùng sample_id trong {split}!'
    assert all(r['error_reason'] is None for r in rows), f'Có lỗi inference trong {split}!'
    print(f'--> {split.capitalize()}: Đủ {len(rows)}/{expected} rows, 0 lỗi.')

print('\n>>> NGHIỆM THU ARTIFACTS VÀ DỰ ĐOÁN THÀNH CÔNG 100% <<<')

In [ ]:
# ===== Cell 8: Đóng gói Artifacts xuất ra /kaggle/working để tải về =====
import os, shutil
from pathlib import Path

out_root = Path('/kaggle/working')

print('--> Đang nén artifacts và runs vào /kaggle/working/...')
!zip -q -r /kaggle/working/efficientad_step4_artifacts.zip artifacts/efficientad/
!zip -q -r /kaggle/working/efficientad_step4_runs.zip runs/efficientad/

art_zip = out_root / 'efficientad_step4_artifacts.zip'
run_zip = out_root / 'efficientad_step4_runs.zip'

assert art_zip.exists(), 'Không tìm thấy file efficientad_step4_artifacts.zip!'
assert run_zip.exists(), 'Không tìm thấy file efficientad_step4_runs.zip!'

print(f'--> [XONG] Đã tạo thành công:')
print(f'   1. {art_zip} ({art_zip.stat().st_size / 2**20:.2f} MB)')
print(f'   2. {run_zip} ({run_zip.stat().st_size / 2**20:.2f} MB)')
print('\nKhi chạy "Save & Run All", bạn có thể tải 2 file này trực tiếp từ tab "Output" của Kaggle Notebook!')